In [ ]:
import odoo
from odoo import fields
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles import numbers
import logging
import warnings
import pytz
logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings('ignore')
logging.getLogger('odoo').setLevel(logging.ERROR)
logging.getLogger('werkzeug').setLevel(logging.ERROR)
odoo.tools.config.parse_config([
    '-c', '/etc/odoo/odoo.conf',
    '-d', 'ZTYRES',
    '--xmlrpc-port=8009',
    '--workers=20'
])
registry = odoo.registry(odoo.tools.config['db_name'])
cr = registry.cursor()
env = odoo.api.Environment(cr, odoo.SUPERUSER_ID, {})

In [ ]:
def convert_to_utc_string(cut_date_str):
    # Ejemplo: cut_date_str = '2025-02-01 05:59:07' (hora local de México)
    mexico_tz = pytz.timezone('America/Mexico_City')
    # Parsear y convertir a UTC
    local_dt = mexico_tz.localize(datetime.strptime(cut_date_str, "%Y-%m-%d %H:%M:%S"))
    utc_dt = local_dt.astimezone(pytz.utc)
    # Retornar como string en formato estándar de Odoo
    return utc_dt.strftime("%Y-%m-%d %H:%M:%S")

df_list = []
cut_date = '2024-12-31 23:59:59'
cut_date = convert_to_utc_string(cut_date)
dates = [
     ('DATA', cut_date)    
]
ValLayer = env['stock.valuation.layer']

for month, date in dates:
    report_data = []
    ids_nacionales = [2, 3, 4, 6, 7, 9, 12, 13, 14, 15, 17]
    ids_importacion = [1, 5, 8, 10, 11, 16, 18, 19, 20, 21, 22, 23]
    domain = [('type', '=', 'product')]
    products_at_date = env['product.product'].with_context(dict({
        'active_model': 'stock.quant',
        'lang': 'es_MX',
        'tz': 'America/Mexico_City',
        'uid': 2,
        'to_date': date,
        'active_test':False,
    })).search(domain)
    for product in products_at_date:
        vals = ValLayer.read_group(
            domain=[
                ('create_date', '<=', cut_date),
                ('product_id', '=', product.id),
            ],
            fields=['value:sum', 'quantity:sum'],
            groupby=['product_id']
        )
        svl_qty = 0
        svl_value = 0
        if vals:
            svl_qty = vals[0].get('quantity',0)
            svl_value = vals[0].get('value',0)
        report_data.append({
        'product_id':product.id,
        'product_name':product.name,
        'product_code':product.default_code,
        'Stock':product.qty_available,
        'Valor Stock': product.standard_price * product.qty_available,
        'Transito': svl_qty-product.qty_available,
        'Valor Transito': (svl_qty-product.qty_available)*product.standard_price,
        'Costo Unitario Base': product.standard_price,
        'Cantidad Valuada': svl_qty,
        'Total Valuado': svl_value
        })
    df = pd.DataFrame(report_data)
    df_list.append((date, df))

In [ ]:

pd.options.display.float_format = '{:,.2f}'.format
import time
import math
filtro = (df['Stock'] != 0)
df_stock = df[filtro]
columns = [
'Transito',
'Valor Transito',
'Cantidad Valuada',
'Total Valuado'
]
df_stock = df_stock.drop(columns=columns)
df_stock.sum(numeric_only=True)


In [ ]:
product_ids = (
    df_stock['product_id']
    .dropna()
    .astype('int64')
    .drop_duplicates()
    .tolist()
)

if not product_ids:
    product_ids = []

MoveLine = env['account.move.line'].with_context(active_test=False)  # incluir archivados
Company = env.company
MXN = env.ref('base.MXN')
USD = env.ref('base.USD')
cutoff_dt = fields.Datetime.from_string(cut_date)  # maneja tz correctamente

def _convert_unit_to(currency_from, amount, currency_to, date_):
    """Convierte un monto unitario de currency_from -> currency_to a la fecha dada, sin redondear."""
    cf = currency_from or Company.currency_id
    return cf._convert(amount, currency_to, Company, date_, round=False)

def _find_last_line(product_id, move_types, cutoff_dt):
    """
    Busca la última línea de AML para el producto:
    1) con date <= cutoff_dt, si existe
    2) si no, la última disponible
    """
    dom_base = [
        ('product_id', '=', product_id),
        ('move_id.state', '=', 'posted'),
        ('display_type', '=', 'product'),           # líneas reales
        ('move_id.move_type', 'in', move_types),
    ]
    dom_cut = dom_base + [('date', '<=', cutoff_dt)]
    line = MoveLine.search(dom_cut, order='date desc', limit=1)
    if line:
        return line
    line = MoveLine.search(dom_base, order='date desc', limit=1)
    return line or env['account.move.line']

def _extract_unit_prices(line):
    """
    Extrae manufacturer, origin, unit_mxn, unit_usd, fecha, tasa_mxn, tasa_usd
    de UNA línea de factura (account.move.line).
    - Prioriza la tasa efectiva de la línea (balance/amount_currency).
    - Si no aplica, cae a currency._convert con la fecha de la factura.
    """
    Company = env.company
    if not line or not line.id:
        return (None, None, None, None, None, None, None)
    # Moneda y fecha "correctas" de la factura
    inv_currency = line.move_id.currency_id or Company.currency_id
    inv_date = line.move_id.invoice_date or line.move_id.date or line.date
    # Precio unitario en la moneda de la factura (como se ve en la línea)
    unit_in_inv_currency = line.price_unit or 0.0
    # --- Tasa efectiva (si hay amount_currency) ---
    # balance: en moneda de la compañía
    # amount_currency: en moneda de la factura
    tasa_inv_to_company = None
    if line.amount_currency and line.quantity:
        # precio unitario real en moneda de factura y en moneda de compañía
        subtotal_inv = line.amount_currency            # total en moneda de la factura (con signo)
        subtotal_comp = line.balance                   # total en moneda de la compañía (con signo)
        # Tasa efectiva usada por Odoo: 1 inv_currency = X company_currency
        tasa_inv_to_company = abs(subtotal_comp) / abs(subtotal_inv)
    # Si no se pudo inferir tasa de la línea, usar la tabla de tasas
    if not tasa_inv_to_company:
        tasa_inv_to_company = inv_currency._convert(
            1.0, Company.currency_id, Company, inv_date, round=False
        )
    # Calcula unitarios en MXN y USD
    # Paso 1: pasa de inv_currency -> moneda de compañía (p.ej. MXN si Company es MX)
    unit_in_company = unit_in_inv_currency * tasa_inv_to_company
    # Paso 2: si la moneda de compañía ya es MXN, unit_mxn = unit_in_company; si no, convertir
    if Company.currency_id == MXN:
        unit_mxn = unit_in_company
    else:
        unit_mxn = Company.currency_id._convert(unit_in_company, MXN, Company, inv_date, round=False)
    # Para USD:
    if inv_currency == USD:
        unit_usd = unit_in_inv_currency
    else:
        unit_usd = inv_currency._convert(unit_in_inv_currency, USD, Company, inv_date, round=False)
    # Tasas auxiliares 1 inv_currency -> MXN / USD (útiles para depurar)
    tasa_mxn = inv_currency._convert(1.0, MXN, Company, inv_date, round=False)
    tasa_usd = inv_currency._convert(1.0, USD, Company, inv_date, round=False)
    manufacturer = getattr(line.product_id.manufacturer_id, 'name', '') or ''
    origin = getattr(line.product_id.manufacturer_id, 'product_nationality', '') or ''
    return (manufacturer, origin, unit_mxn, unit_usd, inv_date, tasa_mxn, tasa_usd)

def _fmt_hms(sec):
    if sec is None or math.isinf(sec) or math.isnan(sec):
        return "--:--"
    sec = int(sec)
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def _progress(idx, total, pid=None, extra=""):
    pct = (idx / total * 100) if total else 0
    msg = f"{idx}/{total} ({pct:5.1f}%)"
    if pid is not None:
        msg += f" | product_id={pid}"
    if extra:
        msg += f" | {extra}"
    print(msg, flush=True)

rows = []
errors = 0
total = len(product_ids)
start = time.time()
last_tick = start
tick_every = 50 

print(f"Iniciando: {total} productos | cut_date={cut_date}", flush=True)

for idx, pid in enumerate(product_ids, start=1):
    try:
        # Última COMPRA
        last_in_line = _find_last_line(pid, move_types=['in_invoice'], cutoff_dt=cutoff_dt)
        manufacturer,origin,pu_mxn, pu_usd, pu_date, tasa_mxn, tasa_usd = _extract_unit_prices(last_in_line)
        # Última VENTA
        last_out_line = _find_last_line(pid, move_types=['out_invoice'], cutoff_dt=cutoff_dt)
        manufacturer,origin,su_mxn, su_usd, su_date,tasa_mxn, tasa_usd = _extract_unit_prices(last_out_line)
        rows.append({
            'product_id': pid,
            'manufacturer': manufacturer,
            'origin': origin,
            'last_purchase_unit_mxn': pu_mxn,
            'last_purchase_unit_usd': pu_usd,
            'last_purchase_date': pu_date,
            'last_sale_unit_mxn': su_mxn,
            'rate_mxn': tasa_mxn,            
            'last_sale_unit_usd': su_usd,
            'rate_usd': tasa_usd,
            'last_sale_date': su_date,
        })
    except Exception as e:
        errors += 1
        rows.append({
            'product_id': pid,
            'manufacturer': None,
            'origin': None,
            'last_purchase_unit_mxn': None,
            'last_purchase_unit_usd': None,
            'last_purchase_date': None,
            'last_sale_unit_mxn': None,
            'rate_mxn': None,    
            'last_sale_unit_usd': None,
            'rate_usd': None,
            'last_sale_date': None,
        })
        _progress(idx, total, pid, extra=f"ERROR: {e}")
    
    if (idx == 1) or (idx % tick_every == 0) or (idx == total):
        now = time.time()
        elapsed = now - start
        rate = idx / elapsed if elapsed > 0 else 0
        remaining = (total - idx) / rate if rate > 0 else None
        extra = f"elapsed={_fmt_hms(elapsed)} | rate={rate:,.2f} prod/s | ETA={_fmt_hms(remaining)} | errores={errors}"
        _progress(idx, total, pid, extra=extra)

In [ ]:
df_prices = pd.DataFrame(rows)
df_result = df_stock.merge(df_prices, on='product_id', how='left')
df_start_value = df_result.copy()

In [ ]:
def reemplazar_costo(row):
    base = row.get('Costo Unitario Base', 0) or 0
    last_purchase = row.get('last_purchase_unit_mxn', 0) or 0
    last_sale = row.get('last_sale_unit_mxn', 0) or 0
    if base > 0:
        return base
    if base <= 0:
        if last_purchase>0:
            return last_purchase
        elif last_sale>0:
            return last_sale/1.30
        else:
            return 0

df_result['Costo Unitario Base'] = df_result.apply(reemplazar_costo, axis=1)

In [ ]:
import numpy as np

df_result['last_value'] = df_result['Costo Unitario Base'] * (
    1.3 * (df_result['origin'].str.lower() == 'imported') + 
    1.0 * (df_result['origin'].str.lower() != 'imported')
)

def calcular_factor(row):
    """
    Calcula el factor de una fila:
    Usa 'last_purchase_unit_mxn' si existe, de lo contrario usa 'last_sale_unit_mxn'.
    Si 'last_value' es 0 o NaN, devuelve 0 (o np.nan si se prefiere).
    """
    last_value = row['last_value']
    purchase = row['last_purchase_unit_mxn']
    sale = row['last_sale_unit_mxn']
    
    # Si no hay valor de referencia, evitar error
    if pd.isna(last_value) or last_value == 0:
        return 0  # o np.nan si prefieres
    
    # Toma purchase si existe, si no, usa sale
    base_unit = purchase if not pd.isna(purchase) else sale
    # Si ambos son NaN, devuelve 0
    if pd.isna(base_unit):
        return 0  # o np.nan
    
    return base_unit / last_value


df_result['factor'] = df_result.apply(calcular_factor, axis=1)
df_result

In [ ]:
with pd.ExcelWriter(f'/mnt/contabilidad/VALORACION 2024/Diciembre.xlsx') as writer:
    df_start_value.to_excel(writer, sheet_name="Valor", index=False)
    df_result.to_excel(writer, sheet_name="Valor+30", index=False)